Adding the libraries

In [16]:
import pandas as pd

Basic statistics of dataset

In [17]:
Full_data = pd.read_csv("iran_poi_raw.csv")
Total_records = Full_data.shape[0]
print()
print(f'Total records of dataset: {Total_records}')

Categories = [
    "amenity" , "shop" , "leisure" , "office" ,
    "tourism" , "craft" , "healthcare"
]
Counts = Full_data["main_key"].value_counts()
for category in Categories:
    count = Counts.get(category , 0)
    precentage = count / Total_records
    print(f"{category} => count: {count} | precentage : {precentage:.2f}")

missing_count = Full_data.isna().sum()
missing_percentage = (missing_count / Total_records) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage (%)": missing_percentage.round(2)
})

print("\nMissing Values Report:")
print(missing_report.to_string())


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3452\3994022593.py:1: DtypeWarning: Columns (0: capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  Full_data = pd.read_csv("iran_poi_raw.csv")



Total records of dataset: 297152
amenity => count: 138493 | precentage : 0.47
shop => count: 72122 | precentage : 0.24
leisure => count: 42320 | precentage : 0.14
office => count: 25972 | precentage : 0.09
tourism => count: 12220 | precentage : 0.04
craft => count: 3911 | precentage : 0.01
healthcare => count: 2114 | precentage : 0.01

Missing Values Report:
                  Missing Values  Missing Percentage (%)
osm_type                       0                    0.00
osm_id                         0                    0.00
version                        0                    0.00
edit_date                      0                    0.00
lat                            0                    0.00
lon                            0                    0.00
main_key                       0                    0.00
main_value                     0                    0.00
name                       73574                   24.76
name:fa                   289881                   97.55
name:en    

City names

In [18]:
cities = Full_data["addr:city"]

unique_cities = cities.nunique(dropna=True)
missing_cities = cities.isna().sum()

print("Unique city values:", unique_cities)
print("Missing city values:", missing_cities)



print("\nMost frequent city values:")
print(cities.value_counts().head(20).to_string())


tehran_variants = [
    "تهران",
    "Tehran",
    "شهر تهران",
    "tehran",
    "TEHRAN"
]

print("\nWritten forms of Tehran:")

for city in tehran_variants:
    count = (cities == city).sum()
    print(f"{city}: {count}")

Unique city values: 2525
Missing city values: 252005

Most frequent city values:
addr:city
تهران        6110
شیراز        4939
اصفهان       1194
مشهد         1125
Tehran       1018
شهر تهران     932
بابل          932
کرج           835
تبریز         821
چهارفریضه     780
فولادشهر      600
کیش           521
اقلید         481
اهواز         479
اردبیل        473
بروجرد        452
tehran        408
قائمشهر       395
بندرعباس      387
رشت           387

Written forms of Tehran:
تهران: 6110
Tehran: 1018
شهر تهران: 932
tehran: 408
TEHRAN: 8


Arabic names

In [19]:
arabic_chars = Full_data["name"].str.contains("[يك]", na=False)

total_names = arabic_chars.sum()
unique_names = Full_data.loc[arabic_chars, "name"].nunique()

print("Total records with Arabic characters:", total_names)
print("Unique names with Arabic characters:", unique_names)


Total records with Arabic characters: 1726
Unique names with Arabic characters: 1697


Capacity value

In [20]:
capacity = Full_data["capacity"].astype("string").str.strip()

missing_value = capacity.isna()

numeric_capacity = pd.to_numeric(capacity, errors="coerce")

invalid = ~missing_value & numeric_capacity.isna()

negative = numeric_capacity < 0


print("Missing capacity values:", missing_value.sum())
print("Valid non-negative numeric values:", (numeric_capacity.notna() & ~negative).sum())
print("Invalid non-numeric values:", invalid.sum())
print("Negative values:", negative.sum())


print("\nNegative capacity values:")
print(Full_data.loc[negative, ["capacity"]].value_counts().to_string())


print("\nInvalid non-numeric values:")
print(Full_data.loc[invalid, ["capacity"]].value_counts().head(10).to_string())

Missing capacity values: 296298
Valid non-negative numeric values: 842
Invalid non-numeric values: 10
Negative values: 2

Negative capacity values:
capacity
-1.0        1
-50.0       1

Invalid non-numeric values:
capacity             
۵ نفر                    1
۵۰                       1
۲۰                       1
۶۰                       1
۱۰                       1
۸۰۰متر مربع              1
مخزن (تانکر) آب حسینی    1
مشهد                     1
ضض300                    1
ترمینال                  1


Phone number

In [25]:
phones = Full_data["phone"].astype("string").str.strip()

phones = phones[phones != ""]

persian_digits = "۰۱۲۳۴۵۶۷۸۹"
arabic_digits = "٠١٢٣٤٥٦٧٨٩"

def get_phone_format(phone):
    has_persian = any(ch in persian_digits for ch in phone)
    has_latin = any(ch in "0123456789" for ch in phone)
    has_arabic = any(ch in arabic_digits for ch in phone)

    if has_latin:
        digit_type = "Latin digits"
    elif has_arabic:
        digit_type = "Arabic digits"
    elif has_persian:
        digit_type = "Persian digits" 
    else:
        digit_type = "No digits"   

    if phone.startswith("+98") or phone.startswith("+۹۸") :
        country_code = "With +98"
    elif phone.startswith("0098") or phone.startswith("۰۰۹۸"):
        country_code = "With 0098"
    elif phone.startswith("98") or phone.startswith("۹۸"):
        country_code = "With 98"
    else:
        country_code = "Without country code"

    spacing = "With spaces" if " " in phone else "Without spaces"

    digits = "".join(
    ch for ch in phone
    if ch in persian_digits + arabic_digits + "0123456789"
)

    if len(digits) < 8:
        length_type = "Possibly incomplete"
    elif 8 < len(digits) < 10:
        length_type = "if it's phone , Incomplete"
    else:
        length_type = "+10 digits"
        
            

    return digit_type, country_code, spacing, length_type


phone_formats = phones.apply(get_phone_format)

phone_table = pd.DataFrame({
    "phone": phones,
    "digit_type": [x[0] for x in phone_formats],
    "country_code": [x[1] for x in phone_formats],
    "spacing": [x[2] for x in phone_formats],
    "length_type": [x[3] for x in phone_formats]
})


examples = phone_table.drop_duplicates(
    subset=["digit_type", "country_code", "spacing", "length_type"]
).head(20)

print(examples.to_string(index=False))


                            phone     digit_type         country_code        spacing                length_type
+98 21 2611 9186;+98 21 2611 9187   Latin digits             With +98    With spaces                 +10 digits
                     +98 21 64451   Latin digits             With +98    With spaces if it's phone , Incomplete
                    +983132205774   Latin digits             With +98 Without spaces                 +10 digits
                       +985134043   Latin digits             With +98 Without spaces if it's phone , Incomplete
                     021-91099109   Latin digits Without country code Without spaces                 +10 digits
                              555   Latin digits Without country code Without spaces        Possibly incomplete
                       +98 96 550   Latin digits             With +98    With spaces        Possibly incomplete
                    +۹۸۲۱۲۲۸۷۷۱۲۲ Persian digits             With +98 Without spaces                 +10

lat / lon

In [ ]:
Full_data["lat_round"] = Full_data["lat"].round(4)
Full_data["lon_round"] = Full_data["lon"].round(4)


duplicate_mask = Full_data.duplicated(
    subset=["lat_round", "lon_round"],
    keep=False
)

duplicate_candidates = Full_data.loc[duplicate_mask].sort_values(
    ["lat_round", "lon_round"]
).copy()


duplicate_groups = duplicate_candidates.groupby(
    ["lat_round", "lon_round"]
)

print("Records with duplicate rounded coordinates:",
      len(duplicate_candidates))

count = 0

for coordinates, group in duplicate_candidates.groupby(
    ["lat_round", "lon_round"]
):
    for value, value_group in group.groupby("main_value"):
        if len(value_group) > 1:
            count += 1
            
duplicate_data = duplicate_candidates[
    duplicate_candidates.duplicated(
        subset=["lat_round", "lon_round", "main_value"],
        keep=False
    )
].copy()

duplicate_data = duplicate_data.sort_values(
    ["lat_round", "lon_round", "main_value"]
)


print("Number of groups with duplicate cordinate and main_value:", count)  

columns_to_show = [
    "lat_round",
    "lon_round",
    "name",
    "name:fa",
    "main_key",
    "main_value",
    "addr:city",
    "osm_id"
]


count = 0

for coordinates, group in duplicate_groups:
    for value, value_group in group.groupby("main_value"):
            if len(value_group) > 1:
                print("=" * 80)
                print("Rounded coordinates:", coordinates)
                print("Main value:", value)

                print(
                    value_group[columns_to_show]
                    .head(10)
                    .to_string(index=False)
                )

                count = count + 1

                if count == 3:
                    break

    if count == 3:
        break
      


Records with duplicate rounded coordinates: 22300
Number of groups with duplicate cordinate and main_value: 3245
Rounded coordinates: (25.2832, 60.6974)
Main value: place_of_worship
 lat_round  lon_round           name name:fa main_key       main_value addr:city     osm_id
   25.2832    60.6974 مسجد امام حسین     NaN  amenity place_of_worship       NaN 7497107222
   25.2832    60.6974            NaN     NaN  amenity place_of_worship       NaN  917649228
Rounded coordinates: (25.2895, 60.6375)
Main value: hotel
 lat_round  lon_round      name name:fa main_key main_value addr:city     osm_id
   25.2895    60.6375 هتل سپیده     NaN  tourism      hotel       NaN 4757824521
   25.2895    60.6375 هتل سپیده     NaN  tourism      hotel       NaN 1137535985
Rounded coordinates: (25.2898, 60.6335)
Main value: park
 lat_round  lon_round name name:fa main_key main_value addr:city     osm_id
   25.2898    60.6335 پارک     NaN  leisure       park       NaN 7436803557
   25.2898    60.6335  NaN     N

Normalize city names

In [ ]:
unique_before = Full_data["addr:city"].nunique()

Full_data["city_normalized"] = (
    Full_data["addr:city"]
    .astype("string")
    .str.strip()
    .str.lower()
)

Full_data["city_normalized"] = Full_data["city_normalized"].str.replace("شهر ", "")

city_mapping = {
    "tehran": "تهران",
    "mashhad": "مشهد",
    "isfahan": "اصفهان",
    "esfahan": "اصفهان",
    "shiraz": "شیراز",
    "tabriz": "تبریز",
    "karaj": "کرج",
    "qom": "قم",
    "ahvaz": "اهواز",
    "ahwaz": "اهواز",
    "kermanshah": "کرمانشاه",
    "urmia": "ارومیه",
    "orumiyeh": "ارومیه",
    "rasht": "رشت",
    "zahedan": "زاهدان",
    "yazd": "یزد",
    "arak": "اراک",
    "bandar abbas": "بندرعباس",
    "hamedan": "همدان",
    "babol": "بابل",
    "ardebil" : "اردبیل",
    "ardabil" : "اردبیل",
    "kish" : "کیش"
}

Full_data["city_normalized"] = Full_data["city_normalized"].replace(
    city_mapping
)

unique_after = Full_data["city_normalized"].nunique()

print("Unique cities before normalization:", unique_before)
print("Unique cities after normalization:", unique_after)
print("Reduction in unique values:", unique_before - unique_after)
print(f"Most frequent cities:")
print(Full_data["city_normalized"].value_counts().head(30).to_string())


Unique cities before normalization: 2525
Unique cities after normalization: 2291
Reduction in unique values: 234
Most frequent cities:
city_normalized
تهران        8476
شیراز        5067
مشهد         1593
اصفهان       1271
بابل          983
تبریز         947
کرج           885
چهارفریضه     780
فولادشهر      600
اهواز         549
کیش           523
قم            505
اردبیل        489
اقلید         481
یزد           472
رشت           470
بروجرد        453
بندرعباس      408
قائمشهر       402
ارومیه        397
گرگان         371
کرمانشاه      347
سمنان         307
قائم شهر      227
بناب          224
بابلسر        219
مراغه         214
ساری          213
کرمان         211
کاشان         208


Convert capacity

In [ ]:
def convert_capacity(value):
    if pd.isna(value):
        return float("Nan")

    value = str(value).strip()

    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"

    for i in range(10):
        value = value.replace(persian_digits[i], english_digits[i])

    try:
        return int(value)
    except ValueError:
        return float("nan")


Full_data["capacity_numeric"] = Full_data["capacity"].apply(convert_capacity)

nan_count = Full_data["capacity_numeric"].isna().sum()

print("Number of NaN records:", nan_count)

Number of NaN records: 296588


Phone numbers normalization

In [ ]:
Full_data["phone_normalized"] = Full_data["phone"].astype("string")

persian_digits = "۰۱۲۳۴۵۶۷۸۹"
english_digits = "0123456789"

for i in range(10):
    Full_data["phone_normalized"] = Full_data["phone_normalized"].str.replace(
        persian_digits[i], english_digits[i]
    )

digit_count = Full_data["phone_normalized"].str.count(r"\d")


invalid_phone = (
    Full_data["phone_normalized"].notna()
    & Full_data["phone_normalized"].str.strip().ne("")
    & (digit_count < 8)
)

print("Number of probably invalid phone numbers:", invalid_phone.sum())

Number of probably invalid phone numbers: 127


Jacard in duplicate cordinates

In [62]:

import json

named_data = duplicate_data[
    duplicate_data["name"].notna()
    & duplicate_data["name"].astype("string").str.strip().ne("")
    & ~duplicate_data["name"].astype("string").str.contains(r"\d", na=False)
].copy()



def get_tag_keys(tags):
    if pd.isna(tags):
        return set()

    try:
        data = json.loads(tags)
        return set(data.keys())
    except (ValueError, TypeError):
        return set()


def get_name_set(name):
    if pd.isna(name):
        return set()

    return set(str(name).lower().split())


def jaccard(set1, set2):
    if len(set1 | set2) == 0:
        return 0

    return len(set1 & set2) / len(set1 | set2)


pairs = []


for group_key, group in named_data.groupby(
    ["lat_round", "lon_round", "main_value"]
):
    group = group.reset_index(drop=True)

   
    for i in range(len(group)):
        for j in range(i + 1, len(group)):

            row1 = group.iloc[i]
            row2 = group.iloc[j]

            tags1 = get_tag_keys(row1["other_tags"])
            tags2 = get_tag_keys(row2["other_tags"])

            name_similarity = jaccard(
                get_name_set(row1["name"]),
                get_name_set(row2["name"])
            )

            tags_similarity = jaccard(tags1, tags2)

            pairs.append({
                "osm_id_1": row1["osm_id"],
                "osm_id_2": row2["osm_id"],
                "name_1": row1["name"],
                "name_2": row2["name"],
                "lat_1": row1["lat_round"],
                "lon_1": row1["lon_round"],
                "lat_2": row2["lat_round"],
                "lon_2": row2["lon_round"],
                "other_tags_jaccard": tags_similarity,
                "name_jaccard": name_similarity
            })


pairs_data = pd.DataFrame(pairs)


# top_pairs = pairs_data.sort_values(
#     "other_tags_jaccard",
#     ascending=True
# ).head(20)
top_pairs = pairs_data.sample(n=20 , random_state=10)

print("Number of records with names:", len(named_data))
print("Number of pairs:", len(pairs_data))

for index, row in top_pairs.reset_index(drop=True).iterrows():
    print("\n" + "=" * 80)
    print("Pair number:", index + 1)
    print("-" * 80)
    print("OSM ID 1:", row["osm_id_1"])
    print("Name 1:", row["name_1"])
    print("Coordinates 1:",f"({row['lat_1']} , {row['lon_1']})")
    print("OSM ID 2:", row["osm_id_2"])
    print("Name 2:", row["name_2"])
    print("Coordinates 2:", f"({row['lat_2']} , {row['lon_2']})")
    print("Other tags Jaccard:", row["other_tags_jaccard"])
    print("Name Jaccard:", row["name_jaccard"])

print("\n" + "=" * 80)



Number of records with names: 4112
Number of pairs: 2291

Pair number: 1
--------------------------------------------------------------------------------
OSM ID 1: 2050321983
Name 1: تاکسی ترمینال صفه
Coordinates 1: (32.5891 , 51.6707)
OSM ID 2: 962338208
Name 2: تاکسی ترمینال صفه
Coordinates 2: (32.5891 , 51.6707)
Other tags Jaccard: 0.0
Name Jaccard: 1.0

Pair number: 2
--------------------------------------------------------------------------------
OSM ID 1: 12064525467
Name 1: نمایندگی کاشی مرجان
Coordinates 1: (29.6255 , 52.514)
OSM ID 2: 13318854866
Name 2: فروشگاه کاشی سرامیک نجیب زاده
Coordinates 2: (29.6255 , 52.514)
Other tags Jaccard: 1.0
Name Jaccard: 0.14285714285714285

Pair number: 3
--------------------------------------------------------------------------------
OSM ID 1: 11339285535
Name 1: وکیل سپیده امیری ابوالوردی
Coordinates 1: (29.622 , 52.5305)
OSM ID 2: 11339285536
Name 2: وکیل مرضیه رنجبر
Coordinates 2: (29.622 , 52.5305)
Other tags Jaccard: 1.0
Name Jaccard: 0

Threshold for duplicate cordinates

In [64]:
name_threshold = 0.6
tags_threshold = 0.6



duplicate_pairs = pairs_data[
    (pairs_data["name_jaccard"] >= name_threshold)
    & (pairs_data["other_tags_jaccard"] >= tags_threshold)
]

print("Number of duplicate pairs:", len(duplicate_pairs))

duplicate_pairs = duplicate_pairs.sort_values(
    "other_tags_jaccard",        
    ascending=True                 
)               
duplicate_pairs = duplicate_pairs.sort_values(
    "name_jaccard",        
    ascending=True                 
)                  


for index, row in duplicate_pairs.reset_index(drop=True).iterrows():
    print("=" * 80)
    print("Pair number:", index + 1)
    print("" + "=" * 80)
    print("Name 1:", row["name_1"])
    print("Name 2:", row["name_2"])
    print("Coordinates 1:", row["lat_1"], row["lon_1"])
    print("Coordinates 2:", row["lat_2"], row["lon_2"])
    print("Name Jaccard:", row["name_jaccard"])
    print("Other tags Jaccard:", row["other_tags_jaccard"])

Number of duplicate pairs: 157
Pair number: 1
Name 1: طلا و جواهری علی
Name 2: طلا و جواهری ترابی
Coordinates 1: 29.6128 52.5426
Coordinates 2: 29.6128 52.5426
Name Jaccard: 0.6
Other tags Jaccard: 0.6666666666666666
Pair number: 2
Name 1: آرایشی و بهداشتی ترانه
Name 2: آرایشی و بهداشتی مروارید
Coordinates 1: 29.6195 52.5297
Coordinates 2: 29.6195 52.5297
Name Jaccard: 0.6
Other tags Jaccard: 1.0
Pair number: 3
Name 1: طلا و جواهر اکسید
Name 2: طلا و جواهر آیناز
Coordinates 1: 29.6126 52.5432
Coordinates 2: 29.6126 52.5432
Name Jaccard: 0.6
Other tags Jaccard: 1.0
Pair number: 4
Name 1: طلا و جواهر آیناز
Name 2: طلا و جواهر تک
Coordinates 1: 29.6126 52.5432
Coordinates 2: 29.6126 52.5432
Name Jaccard: 0.6
Other tags Jaccard: 1.0
Pair number: 5
Name 1: طلا و جواهر اکسید
Name 2: طلا و جواهر تک
Coordinates 1: 29.6126 52.5432
Coordinates 2: 29.6126 52.5432
Name Jaccard: 0.6
Other tags Jaccard: 1.0
Pair number: 6
Name 1: رادیولوژی دکتر کیمیا محمدی
Name 2: ماموگرافی دکتر کیمیا محمدی
Coordina

Mode and Frequency

In [76]:
frequency = Full_data.groupby(["main_key", "main_value"]).size()

for key in Full_data["main_key"].unique():
    data = frequency.loc[key]
    print("=" * 80)
    print("Main key:", key)
    print("Mode:", data.idxmax())
    print("Mode frequency:", data.max())

Main key: amenity
Mode: place_of_worship
Mode frequency: 17810
Main key: office
Mode: government
Mode frequency: 9389
Main key: shop
Mode: supermarket
Mode frequency: 7667
Main key: tourism
Mode: hotel
Mode frequency: 2577
Main key: leisure
Mode: park
Mode frequency: 20097
Main key: healthcare
Mode: laboratory
Mode frequency: 957
Main key: craft
Mode: photographer
Mode frequency: 366


Hub km and Nearest hub

In [87]:

hub_data = Full_data["hub_km"].dropna()
print(len(hub_data))

count = (hub_data > 300).sum()
percentage = count / len(hub_data) * 100

print("Number of records over 300 km:", count)
print("Percentage over 300 km:", f"{percentage} %")



hub_data = sorted(hub_data)

n = len(hub_data)

p25 = hub_data[int(0.25 * (n - 1))]
p50 = hub_data[int(0.50 * (n - 1))]
p75 = hub_data[int(0.75 * (n - 1))]
p90 = hub_data[int(0.90 * (n - 1))]

print("\nPercentiles of hub_km:")
print("25th percentile:", p25)
print("50th percentile:", p50)
print("75th percentile:", p75)
print("90th percentile:", p90)

far_places = Full_data[Full_data["hub_km"] > 300]

print("\nMost common cities for records over 300 km:")
print(far_places["nearest_hub"].value_counts().head(10).to_string())

print("\nAverage coordinates of records over 300 km:")
print("Latitude:", far_places["lat"].mean())
print("Longitude:", far_places["lon"].mean())


297152
Number of records over 300 km: 2300
Percentage over 300 km: 0.7740146457032092 %

Percentiles of hub_km:
25th percentile: 5.0
50th percentile: 14.99
75th percentile: 69.98
90th percentile: 120.95

Most common cities for records over 300 km:
nearest_hub
Zahedan        1784
BandarAbbas     515
Ardabil           1

Average coordinates of records over 300 km:
Latitude: 25.950330324347824
Longitude: 60.63325906999999
